# LSTM with Bahdanau Attention: English to Portuguese Translation

## Overview

This tutorial demonstrates how to build a **Sequence-to-Sequence (Seq2Seq)** model for machine translation using:
- **LSTM (Long Short-Term Memory)** networks for encoding and decoding sequences
- **Bahdanau Attention Mechanism** to help the model focus on relevant parts of the input when generating output

### Key Concepts:
1. **LSTM**: A type of recurrent neural network that can learn long-term dependencies in sequential data
2. **Attention**: Allows the decoder to focus on different parts of the input sequence dynamically
3. **Encoder-Decoder Architecture**: Input is encoded into a context vector, which the decoder uses to generate output
4. **Teacher Forcing**: During training, the true target tokens are fed as input instead of predicted ones

### Dataset:
We'll use a Portuguese-English sentence pairs dataset to train an English-to-Portuguese translator.

## Step 1: Import Libraries and Configure Parameters

Let's start by importing all necessary libraries and defining hyperparameters that control model behavior.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Embedding, LSTM, Dense, Layer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

/opt/anaconda3/envs/tensorflow/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


TensorFlow version: 2.21.0
GPU Available: []


### Configuration Parameters

These hyperparameters control the model's architecture and training process:
- **FILE_PATH**: Path to the dataset containing English-Portuguese sentence pairs
- **NUM_EXAMPLES**: Number of sentence pairs to use (smaller values for faster training)
- **BATCH_SIZE**: Number of sequences processed in one training step
- **EMBEDDING_DIM**: Dimensionality of word embeddings (maps words to dense vectors)
- **UNITS**: Number of LSTM units (controls model capacity)
- **EPOCHS**: Number of complete passes through the training dataset

In [2]:
# ==========================================
# CONFIGURATION
# ==========================================

FILE_PATH = "./por.txt"
NUM_EXAMPLES = 5000  # Adjust depending on hardware capacity
BATCH_SIZE = 64
EMBEDDING_DIM = 256  # Dimension of word embeddings
UNITS = 512  # Number of LSTM units
EPOCHS = 10

print(f"Configuration:")
print(f"  Max examples: {NUM_EXAMPLES}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Embedding dimension: {EMBEDDING_DIM}")
print(f"  LSTM units: {UNITS}")
print(f"  Epochs: {EPOCHS}")

Configuration:
  Max examples: 5000
  Batch size: 64
  Embedding dimension: 256
  LSTM units: 512
  Epochs: 10


## Step 2: Data Preprocessing Functions

Before training, we need to clean and normalize the text data. This includes:
- Lowercasing and removing extra whitespace
- Handling punctuation
- Removing special characters while preserving accented Portuguese characters
- Adding `<start>` and `<end>` tokens to delimit sequences

In [3]:
def preprocess_sentence(s):
    """
    Preprocess a sentence for the model.
    
    Steps:
    1. Convert to lowercase
    2. Add spaces around punctuation (e.g., "boy." -> "boy .")
    3. Remove multiple spaces
    4. Keep only letters, punctuation, and accented characters
    5. Add <start> and <end> tokens
    
    Args:
        s (str): Input sentence
    
    Returns:
        str: Preprocessed sentence
    """
    # Convert to lowercase and strip whitespaces
    s = s.lower().strip()
    
    # Add space between words and punctuation marks
    # e.g., "he is a boy." -> "he is a boy ."
    s = re.sub(r"([?.!,¿])", r" \1 ", s)
    s = re.sub(r'[" "]+', " ", s)
    
    # Replace everything except (a-z, A-Z, ".", "?", "!", ",")
    # Also keep accented characters for Portuguese
    s = re.sub(r"[^a-zA-Z?.!,¿áéíóúâêôãõçÀÉÍÓÚÂÊÔÃÕÇ]+", " ", s)
    s = s.strip()
    
    # Add start and end tokens
    s = "<start> " + s + " <end>"
    return s

# Test the preprocessing function
test_sentence = "Hello! This is a test."
print(f"Original: {test_sentence}")
print(f"Preprocessed: {preprocess_sentence(test_sentence)}")

Original: Hello! This is a test.
Preprocessed: <start> hello ! this is a test . <end>


## Step 3: Load and Prepare Dataset

The dataset is stored as tab-separated values (TSV) with English and Portuguese sentences on each line.
We'll:
1. Load the file
2. Split into English and Portuguese sentences
3. Preprocess each sentence
4. Convert text to numerical sequences (tokenization)
5. Pad sequences to uniform length

In [4]:
def load_dataset(path, num_examples):
    """
    Load English-Portuguese sentence pairs from a file.
    
    Args:
        path (str): Path to the dataset file
        num_examples (int): Maximum number of examples to load
    
    Returns:
        tuple: (english_sentences, portuguese_sentences)
    """
    lines = open(path, encoding="utf-8").read().strip().split("\n")

    en_sentences = []
    pt_sentences = []

    # Process each line (tab-separated English and Portuguese)
    for line in lines[:num_examples]:
        parts = line.split("\t")
        if len(parts) >= 2:
            # parts[0] = English (source), parts[1] = Portuguese (target)
            # You can swap these if you want to translate Portuguese -> English
            en_sentences.append(preprocess_sentence(parts[0]))
            pt_sentences.append(preprocess_sentence(parts[1]))

    return en_sentences, pt_sentences

print("Loading and preprocessing dataset...")
input_lang, target_lang = load_dataset(FILE_PATH, NUM_EXAMPLES)
print(f"Loaded {len(input_lang)} sentence pairs")
print(f"\nFirst example:")
print(f"  English: {input_lang[0]}")
print(f"  Portuguese: {target_lang[0]}")

Loading and preprocessing dataset...
Loaded 5000 sentence pairs

First example:
  English: <start> go . <end>
  Portuguese: <start> vai . <end>


### Tokenization and Padding

**Tokenization** converts words to integers (vocabulary indices), and **padding** ensures all sequences have the same length.

- **Tokenizer**: Maps each unique word to an integer ID
- **texts_to_sequences**: Converts text to sequences of integer IDs
- **pad_sequences**: Adds zeros to make all sequences the same length

In [5]:
def tokenize(lang):
    """
    Tokenize text sequences and pad them to uniform length.
    
    Args:
        lang (list): List of text sentences
    
    Returns:
        tuple: (padded_tensor, tokenizer)
    """
    # Create tokenizer with no filters to preserve <start> and <end> tokens
    lang_tokenizer = Tokenizer(filters="")
    lang_tokenizer.fit_on_texts(lang)
    
    # Convert text to sequences of integers
    tensor = lang_tokenizer.texts_to_sequences(lang)
    
    # Pad sequences to same length (post-padding: zeros at the end)
    tensor = pad_sequences(tensor, padding="post")
    
    return tensor, lang_tokenizer

# Tokenize both input and target languages
input_tensor, input_tokenizer = tokenize(input_lang)
target_tensor, target_tokenizer = tokenize(target_lang)

# Get dataset statistics
max_length_input = input_tensor.shape[1]
max_length_target = target_tensor.shape[1]
vocab_input_size = len(input_tokenizer.word_index) + 1
vocab_target_size = len(target_tokenizer.word_index) + 1

print(f"Total English vocabulary size: {vocab_input_size}")
print(f"Total Portuguese vocabulary size: {vocab_target_size}")
print(f"Max length input: {max_length_input}, Max length target: {max_length_target}")
print(f"\nExample tokenized sequence:")
print(f"  Original: {input_lang[0]}")
print(f"  Tokenized: {input_tensor[0][:20]}...")  # Show first 20 tokens

Total English vocabulary size: 1208
Total Portuguese vocabulary size: 2197
Max length input: 8, Max length target: 11

Example tokenized sequence:
  Original: <start> go . <end>
  Tokenized: [ 1 23  3  2  0  0  0  0]...


### Create TensorFlow Dataset

Convert numpy arrays to a TensorFlow dataset for efficient batch processing during training.

In [6]:
# Create tf.data Dataset for efficient batch processing
dataset = tf.data.Dataset.from_tensor_slices((input_tensor, target_tensor))
dataset = dataset.shuffle(len(input_tensor)).batch(BATCH_SIZE, drop_remainder=True)

print(f"Dataset created with {len(dataset)} batches")
print(f"Each batch shape: (input: {BATCH_SIZE} x {max_length_input}, target: {BATCH_SIZE} x {max_length_target})")

Dataset created with 78 batches
Each batch shape: (input: 64 x 8, target: 64 x 11)


## Step 4: Model Architecture - Encoder

### What is the Encoder?

The **Encoder** reads the input sequence and produces:
- **Encoder output**: All hidden states from each time step
- **Encoder hidden state**: Final hidden state (used to initialize decoder)

### Architecture:
1. **Embedding Layer**: Converts word IDs to dense vectors (e.g., 256-dimensional)
2. **LSTM Layer**: Processes the sequence and maintains hidden state
   - Input: Embedded word vectors
   - Output: Sequence of hidden states (for attention), final hidden state (h, c)

### LSTM States:
- **h (hidden state)**: Short-term memory
- **c (cell state)**: Long-term memory

In [7]:
class Encoder(tf.keras.Model):
    """
    Encoder model that reads input sequence and produces encoder outputs and hidden states.
    """
    
    def __init__(self, vocab_size, embedding_dim, enc_units, batch_sz):
        """
        Initialize the Encoder.
        
        Args:
            vocab_size (int): Size of vocabulary
            embedding_dim (int): Dimension of word embeddings
            enc_units (int): Number of LSTM units
            batch_sz (int): Batch size
        """
        super(Encoder, self).__init__()
        self.batch_sz = batch_sz
        self.enc_units = enc_units
        
        # Embedding layer: maps word IDs (integers) to dense vectors
        # Output shape: (batch_size, sequence_length, embedding_dim)
        self.embedding = Embedding(vocab_size, embedding_dim)
        
        # LSTM layer:
        # return_sequences=True: returns all hidden states (needed for attention)
        # return_state=True: returns final hidden state and cell state
        self.lstm = LSTM(
            enc_units,
            return_sequences=True,
            return_state=True,
            recurrent_initializer="glorot_uniform",
        )

    def call(self, x, hidden):
        """
        Forward pass through encoder.
        
        Args:
            x: Input tensor (batch_size, sequence_length)
            hidden: Initial hidden states [h, c]
        
        Returns:
            output: All hidden states (batch_size, sequence_length, enc_units)
            [state_h, state_c]: Final hidden and cell states
        """
        # Embed the input
        x = self.embedding(x)
        
        # Pass through LSTM
        # output: all hidden states from each time step
        # state_h, state_c: final hidden and cell states
        output, state_h, state_c = self.lstm(x, initial_state=hidden)
        
        return output, [state_h, state_c]

    def initialize_hidden_state(self):
        """
        Initialize hidden and cell states to zeros.
        
        Returns:
            list: [h (zeros), c (zeros)]
        """
        return [
            tf.zeros((self.batch_sz, self.enc_units)),
            tf.zeros((self.batch_sz, self.enc_units)),
        ]

print("Encoder class defined successfully!")

Encoder class defined successfully!


## Step 5: Attention Mechanism - Bahdanau Attention

### What is Attention?

Instead of using only the final hidden state, **attention** allows the decoder to focus on different parts of the input sequence.

### How Bahdanau Attention Works:

1. **Score Calculation**: For each encoder hidden state, compute a score showing relevance
   - `score = V^T * tanh(W1 * query + W2 * values)`
   - query: decoder's current hidden state
   - values: all encoder hidden states

2. **Attention Weights**: Apply softmax to convert scores to probabilities (sum to 1)

3. **Context Vector**: Weighted sum of encoder hidden states
   - Higher attention weights = more focus on those hidden states

### Intuition:
When translating each word, the model learns to "attend to" the most relevant input words, similar to how human translators focus on specific parts of a sentence.

In [8]:
class BahdanauAttention(Layer):
    """
    Bahdanau Attention mechanism.
    Computes attention weights and context vector based on decoder query and encoder values.
    """
    
    def __init__(self, units):
        """
        Initialize attention layers.
        
        Args:
            units (int): Number of attention units (typically same as LSTM units)
        """
        super(BahdanauAttention, self).__init__()
        
        # Learnable weight matrices
        self.W1 = Dense(units)  # For decoder query
        self.W2 = Dense(units)  # For encoder values
        self.V = Dense(1)       # Final projection to scalar score

    def call(self, query, values):
        """
        Compute attention weights and context vector.
        
        Args:
            query: Decoder's hidden state (batch_size, hidden_size)
            values: Encoder's output (batch_size, max_length, hidden_size)
        
        Returns:
            context_vector: Weighted sum of values (batch_size, hidden_size)
            attention_weights: Normalized attention scores (batch_size, max_length, 1)
        """
        
        # Expand query dimension for broadcasting
        # From: (batch_size, hidden_size)
        # To:   (batch_size, 1, hidden_size)
        query_with_time_axis = tf.expand_dims(query, 1)

        # Calculate attention scores
        # score = V^T * tanh(W1 * query + W2 * values)
        # This is computed for each encoder position
        score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
        # score shape: (batch_size, max_length, 1)

        # Apply softmax to get attention weights (probabilities)
        # These weights sum to 1 across the sequence dimension
        attention_weights = tf.nn.softmax(score, axis=1)
        # attention_weights shape: (batch_size, max_length, 1)

        # Compute context vector as weighted sum of encoder outputs
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)
        # context_vector shape: (batch_size, hidden_size)

        return context_vector, attention_weights

print("BahdanauAttention class defined successfully!")

BahdanauAttention class defined successfully!


## Step 6: Model Architecture - Decoder with Attention

### What is the Decoder?

The **Decoder** generates the output sequence step-by-step using:
- The **context vector** from attention (what to focus on)
- The **previous hidden state** (what was generated before)
- The **current input token** (either true or predicted)

### Architecture:
1. **Embedding Layer**: Converts target word IDs to vectors
2. **Attention Mechanism**: Computes context vector from encoder outputs
3. **Concatenation**: Combines context vector with embedded current word
4. **LSTM Layer**: Processes concatenated input and updates hidden state
5. **Output Dense Layer**: Projects to vocabulary size for prediction

### Teacher Forcing:
During training, we feed the true target tokens as input (instead of predictions) to help the model learn faster.

In [9]:
class Decoder(tf.keras.Model):
    """
    Decoder model with Bahdanau attention.
    Generates output sequence step-by-step using encoder outputs and attention.
    """
    
    def __init__(self, vocab_size, embedding_dim, dec_units, batch_sz):
        """
        Initialize the Decoder.
        
        Args:
            vocab_size (int): Size of target vocabulary
            embedding_dim (int): Dimension of word embeddings
            dec_units (int): Number of LSTM units
            batch_sz (int): Batch size
        """
        super(Decoder, self).__init__()
        self.batch_sz = batch_sz
        self.dec_units = dec_units
        
        # Embedding layer for target language
        self.embedding = Embedding(vocab_size, embedding_dim)
        
        # LSTM layer
        self.lstm = LSTM(
            dec_units,
            return_sequences=True,
            return_state=True,
            recurrent_initializer="glorot_uniform",
        )
        
        # Output layer: projects to vocabulary size
        self.fc = Dense(vocab_size)

        # Attention mechanism
        self.attention = BahdanauAttention(self.dec_units)

    def call(self, x, hidden, enc_output):
        """
        Forward pass through decoder for one time step.
        
        Args:
            x: Current input token (batch_size, 1)
            hidden: Previous hidden states [h, c]
            enc_output: All encoder hidden states
        
        Returns:
            x: Predictions for all vocabulary items (batch_size, vocab_size)
            [state_h, state_c]: Updated hidden and cell states
            attention_weights: Attention weights for visualization
        """
        
        # Step 1: Compute context vector using attention
        # hidden[0] is the hidden state h (not the cell state c)
        context_vector, attention_weights = self.attention(hidden[0], enc_output)

        # Step 2: Embed the current input token
        # x shape: (batch_size, 1, embedding_dim)
        x = self.embedding(x)

        # Step 3: Concatenate context vector with embedded token
        # This gives the LSTM more information to work with
        # x shape becomes: (batch_size, 1, embedding_dim + dec_units)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)

        # Step 4: Pass through LSTM
        # output shape: (batch_size, 1, dec_units)
        output, state_h, state_c = self.lstm(x, initial_state=hidden)

        # Step 5: Project to vocabulary size
        output = tf.reshape(output, (-1, output.shape[2]))
        # x shape: (batch_size, vocab_size)
        x = self.fc(output)

        return x, [state_h, state_c], attention_weights

print("Decoder class defined successfully!")

Decoder class defined successfully!


## Step 7: Initialize Models

Create encoder and decoder instances with the calculated vocabulary sizes and configured hyperparameters.

In [10]:
# Initialize Models
encoder = Encoder(vocab_input_size, EMBEDDING_DIM, UNITS, BATCH_SIZE)
decoder = Decoder(vocab_target_size, EMBEDDING_DIM, UNITS, BATCH_SIZE)

print(f"Encoder initialized:")
print(f"  Input vocabulary: {vocab_input_size}")
print(f"  Embedding dimension: {EMBEDDING_DIM}")
print(f"  LSTM units: {UNITS}")
print(f"\nDecoder initialized:")
print(f"  Output vocabulary: {vocab_target_size}")
print(f"  Embedding dimension: {EMBEDDING_DIM}")
print(f"  LSTM units: {UNITS}")

Encoder initialized:
  Input vocabulary: 1208
  Embedding dimension: 256
  LSTM units: 512

Decoder initialized:
  Output vocabulary: 2197
  Embedding dimension: 256
  LSTM units: 512


## Step 8: Loss Function and Optimizer

### Loss Function:
- **Sparse Categorical Cross-Entropy**: Suitable when targets are integer class indices
- **Masking**: We mask out padding tokens (0s) so they don't contribute to the loss

### Optimizer:
- **Adam**: Adaptive learning rate optimizer, works well for sequence models

### Why Masking?
Padding tokens don't carry meaning, so we shouldn't penalize the model for incorrect predictions on them.

In [11]:
# ==========================================
# LOSS & OPTIMIZATION
# ==========================================

optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction="none"
)

def loss_function(real, pred):
    """
    Compute loss while masking padding tokens.
    
    Args:
        real: True target sequences (batch_size, vocab_size)
        pred: Predicted sequences (batch_size, vocab_size)
    
    Returns:
        float: Masked loss (excluding padding tokens)
    """
    # Create mask for non-padding tokens (tokens != 0)
    mask = tf.math.not_equal(real, 0)
    
    # Compute cross-entropy loss for all tokens
    loss_ = loss_object(real, pred)

    # Apply mask: set padding token losses to 0
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask

    # Return average loss (average over non-padding tokens)
    return tf.reduce_mean(loss_)

print("Loss function and optimizer configured!")

Loss function and optimizer configured!


## Step 9: Training Loop

### Training Process:

1. **Forward Pass**: 
   - Encode input sequence
   - Use encoder's final hidden state to initialize decoder
   - Iteratively decode, generating one token at a time

2. **Teacher Forcing**:
   - Feed true target tokens as input (not predictions)
   - Helps model learn faster and more stably

3. **Loss Computation**:
   - Accumulate loss across all decoder time steps
   - Mask padding tokens

4. **Backpropagation**:
   - Compute gradients using GradientTape
   - Update weights using optimizer

### Key Insight:
The `@tf.function` decorator compiles the training step into a TensorFlow graph for better performance.

In [12]:
@tf.function
def train_step(inp, targ, enc_hidden):
    """
    Execute one training step (one batch).
    
    Args:
        inp: Input sequence batch (batch_size, seq_length)
        targ: Target sequence batch (batch_size, seq_length)
        enc_hidden: Initial encoder hidden state
    
    Returns:
        float: Average batch loss
    """
    loss = 0

    # Record operations for automatic differentiation
    with tf.GradientTape() as tape:
        # Step 1: Encode the input sequence
        enc_output, enc_hidden = encoder(inp, enc_hidden)
        
        # Step 2: Initialize decoder with encoder's final hidden state
        dec_hidden = enc_hidden
        
        # Step 3: Prepare decoder input - all batches start with <start> token
        dec_input = tf.expand_dims(
            [target_tokenizer.word_index["<start>"]] * BATCH_SIZE, 1
        )

        # Step 4: Decode step-by-step using teacher forcing
        # Skip first token (which is always <start>)
        for t in range(1, targ.shape[1]):
            # Decode one step
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
            
            # Compute loss for this time step
            loss += loss_function(targ[:, t], predictions)
            
            # Teacher forcing: use true target token as next input
            dec_input = tf.expand_dims(targ[:, t], 1)

    # Step 5: Compute average loss
    batch_loss = loss / int(targ.shape[1])
    
    # Step 6: Backpropagation
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))

    return batch_loss

print("Training step function defined!")

Training step function defined!


## Step 10: Execute Training

Train the model for the specified number of epochs. Progress will be printed for every 100 batches.

In [13]:
print("\nStarting Training...\n")

for epoch in range(EPOCHS):
    # Initialize encoder hidden state for new epoch
    enc_hidden = encoder.initialize_hidden_state()
    total_loss = 0

    # Iterate through all batches
    for batch, (inp, targ) in enumerate(dataset):
        batch_loss = train_step(inp, targ, enc_hidden)
        total_loss += batch_loss

        # Print progress every 100 batches
        if batch % 100 == 0:
            print(f"Epoch {epoch + 1} Batch {batch} Loss {batch_loss.numpy():.4f}")

    # Print epoch summary
    print(f"Epoch {epoch + 1} Complete. Total Loss: {total_loss / (batch+1):.4f}\n")

print("Training completed!")


Starting Training...

Epoch 1 Batch 0 Loss 3.2357
Epoch 1 Complete. Total Loss: 1.8284

Epoch 2 Batch 0 Loss 1.5362
Epoch 2 Complete. Total Loss: 1.4904

Epoch 3 Batch 0 Loss 1.3886
Epoch 3 Complete. Total Loss: 1.3219

Epoch 4 Batch 0 Loss 1.1301
Epoch 4 Complete. Total Loss: 1.1675

Epoch 5 Batch 0 Loss 1.1218
Epoch 5 Complete. Total Loss: 1.0313

Epoch 6 Batch 0 Loss 0.9778
Epoch 6 Complete. Total Loss: 0.9266

Epoch 7 Batch 0 Loss 0.8549
Epoch 7 Complete. Total Loss: 0.8385

Epoch 8 Batch 0 Loss 0.7328
Epoch 8 Complete. Total Loss: 0.7632

Epoch 9 Batch 0 Loss 0.6288
Epoch 9 Complete. Total Loss: 0.7016

Epoch 10 Batch 0 Loss 0.5770
Epoch 10 Complete. Total Loss: 0.6449

Training completed!


## Step 11: Evaluation and Translation

### Decoding Modes:

1. **Greedy Decoding** (used here):
   - At each step, select the token with highest probability
   - Fast but may miss better sequences

2. **Beam Search** (not implemented here):
   - Keep top-K candidate sequences
   - More accurate but slower

### Inference Process:
1. Preprocess and tokenize input sentence
2. Encode input to get attention context
3. Decode sequentially:
   - Start with `<start>` token
   - Generate one word at a time
   - Use model's prediction as next input
   - Stop at `<end>` token or max length

In [14]:
def evaluate(sentence):
    """
    Translate a single sentence from input language to target language.
    
    Args:
        sentence (str): Input sentence in English
    
    Returns:
        tuple: (translated_sentence, preprocessed_input)
    """
    # Preprocess input sentence
    sentence = preprocess_sentence(sentence)

    # Tokenize: convert words to indices
    inputs = [
        input_tokenizer.word_index[i]
        for i in sentence.split(" ")
        if i in input_tokenizer.word_index
    ]
    # Pad to encoder's expected length
    inputs = pad_sequences([inputs], maxlen=max_length_input, padding="post")
    inputs = tf.convert_to_tensor(inputs)

    result = ""

    # Initialize encoder hidden state for batch size 1
    hidden = [tf.zeros((1, UNITS)), tf.zeros((1, UNITS))]
    
    # Encode input
    enc_out, enc_hidden = encoder(inputs, hidden)

    # Initialize decoder with encoder's hidden state
    dec_hidden = enc_hidden
    
    # Start with <start> token
    dec_input = tf.expand_dims([target_tokenizer.word_index["<start>"]], 0)

    # Decode step-by-step
    for t in range(max_length_target):
        # Generate predictions
        predictions, dec_hidden, attention_weights = decoder(
            dec_input, dec_hidden, enc_out
        )

        # Select token with highest probability (greedy decoding)
        predicted_id = tf.argmax(predictions[0]).numpy()

        # Stop if we reach <end> token
        if target_tokenizer.index_word[predicted_id] == "<end>":
            return result, sentence

        # Add predicted word to result
        result += target_tokenizer.index_word[predicted_id] + " "

        # Use predicted token as next input (not teacher forcing)
        dec_input = tf.expand_dims([predicted_id], 0)

    return result, sentence


def translate(sentence):
    """
    Translate a sentence and print results.
    
    Args:
        sentence (str): Input sentence to translate
    """
    result, sentence = evaluate(sentence)
    print(f"Input: {sentence}")
    print(f"Predicted translation: {result}\n")

print("Evaluation functions defined!")

Evaluation functions defined!


## Step 12: Test Translation

Now let's test the model with some example sentences to see how well it translates!

In [15]:
# Test with example sentences
print("Testing translation on example sentences:\n")

test_sentences = [
    "Hello!",
    "How are you?",
    "I love machine learning.",
]

for sent in test_sentences:
    translate(sent)

Testing translation on example sentences:

Input: <start> hello ! <end>
Predicted translation: acorda ! 

Input: <start> how are you ? <end>
Predicted translation: como vai ? 

Input: <start> i love machine learning . <end>
Predicted translation: eu me rendo . 



## Summary: What We've Learned

### Architecture Components:

1. **Encoder**: 
   - Processes input sequence
   - Produces context through all hidden states
   - Passes final hidden state to decoder

2. **Attention Mechanism**:
   - Allows decoder to focus on relevant input parts
   - Computes context vector as weighted sum
   - Improves translation quality, especially for longer sentences

3. **Decoder**:
   - Generates output token-by-token
   - Uses attention for better context awareness
   - Concatenates context with embeddings for richer input

### Training Strategy:
- **Teacher Forcing**: Feeding true outputs during training for faster learning
- **Masking**: Ignoring padding tokens in loss computation
- **Gradient Descent**: Backpropagation through the entire encoder-decoder-attention pipeline

### Key Improvements:
- Attention enables the model to handle longer sequences
- Bahdanau attention is computatively efficient compared to alternatives
- Masking ensures padding doesn't hurt model performance

### Next Steps:
- Experiment with different hyperparameters (embedding dimension, LSTM units, batch size)
- Implement **beam search** for better translations
- Try **self-attention** or **transformer** models
- Add **visualization** of attention weights to understand model behavior